# Integrantes


*   Eduardo Souza - (esg3@cesar.school)
*   Alexander Bandeira - (abl3@cesar.school)



# Requisitos

Tecnologias:
A equipe pode escolher trabalhar com Dask ou Spark em cada resposta, não precisa responder tudo com a mesma tecnologia. Se responder a pergunta duas vezes com Dask e Spark, a melhor solução será considerada para compor a nota da resposta.

Escolha do dataset: https://www.kaggle.com/datasets/yasserh/instacart-online-grocery-basket-analysis-dataset/data?select=order_products__prior.csv

Itens avaliados:
- Quantidade de perguntas
- Complexidade das perguntas
- Coerência da solução em relação a pergunta

Itens que diminuem a pontuação:
- Performance
- Exceptions
- Estouro de memória em potencial

Escolher um integrante da equipe para entregar o notebook do Jupyter com os itens abaixo:
- Identificar o notebook adicionando o primeiro nome e um sobrenome (ex: trabalho-AndersonNeves.ipynb) da pessoa responsável por enviar o notebook.
- Nome e sobrenome das pessoas da equipe
- Link para download dos dados originais
- Instruções de quais arquivos serão utilizados
- Código de limpeza, transformação e merge de dados (Opcional)
- 4 perguntas sobre o domínio dos dados
- 4 respostas calculadas utilizando processamento de dados em larga escala com Dask ou Spark

# Sobre o Dataset

Seja seguindo listas de compras meticulosamente planejadas ou deixando o capricho guiar suas escolhas, nossos rituais alimentares únicos definem quem somos. O Instacart, um aplicativo de compras e entregas de supermercado, tem como objetivo facilitar o abastecimento da sua geladeira e despensa com seus produtos favoritos e itens básicos, quando você precisar. Após selecionar os produtos pelo aplicativo Instacart, compradores pessoais revisam seu pedido e fazem as compras na loja e a entrega para você.

# Instalando Dependencias

In [1]:
#!pip list | grep dask
#!pip install "dask[distributed]==2026.1.2"
#!pip install kagglehub[pandas-datasets]

# Importando Libs

In [2]:
import gc
import kagglehub
import dask.dataframe as dd
import pandas as pd
from kagglehub import KaggleDatasetAdapter
from dask.distributed import Client, LocalCluster
from itertools import combinations
from collections import Counter
import logging
import dask
from distributed import wait

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Instanciando Client e ClusterLocal

In [3]:
# Lido na criação dos workers: inicia spill/pause antes do limite rígido (evita picos e OOM).
dask.config.set(
    {
        "distributed.worker.memory.target": 0.55,
        "distributed.worker.memory.spill": 0.75,
        "distributed.worker.memory.pause": 0.85,
        "distributed.worker.memory.terminate": 0.90,
    }
)

cluster = LocalCluster(
    n_workers=4,
    threads_per_worker=2,
    memory_limit="4GB",
    dashboard_address=":8787",
)

client = Client(cluster)

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 14.90 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:55107,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:55118,Total threads: 2
Dashboard: http://127.0.0.1:55120/status,Memory: 3.73 GiB
Nanny: tcp://127.0.0.1:55110,


# Carregando Dataset

Relacional entre produtos e lista de pedidos, através do product_id

In [4]:
products_path = "products.csv"
order_path = "order_products__prior.csv"
dataset_id = "yasserh/instacart-online-grocery-basket-analysis-dataset"

df_products = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    dataset_id,
    "products.csv",
)

df_order_items = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    dataset_id,
    "order_products__prior.csv",
)

## Produtos:

In [5]:
df_products.head()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [6]:
df_products.info()

<class 'pandas.DataFrame'>
RangeIndex: 49688 entries, 0 to 49687
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   product_id     49688 non-null  int64
 1   product_name   49688 non-null  str  
 2   aisle_id       49688 non-null  int64
 3   department_id  49688 non-null  int64
dtypes: int64(3), str(1)
memory usage: 3.0 MB


In [7]:
df_products.describe()

,product_id,aisle_id,department_id
count,49688.000000,49688.000000,49688.000000
mean,24844.500000,67.769582,11.728687
std,14343.834425,38.316162,5.850410
min,1.000000,1.000000,1.000000
25%,12422.750000,35.000000,7.000000
50%,24844.500000,69.000000,13.000000
75%,37266.250000,100.000000,17.000000
max,49688.000000,134.000000,21.000000


## Lista de Pedidos:

In [8]:
df_order_items.head()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [9]:
df_order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 32434489 entries, 0 to 32434488
Data columns (total 4 columns):
 #   Column             Dtype
---  ------             -----
 0   order_id           int64
 1   product_id         int64
 2   add_to_cart_order  int64
 3   reordered          int64
dtypes: int64(4)
memory usage: 989.8 MB


In [10]:
df_order_items.describe()

,order_id,product_id,add_to_cart_order,reordered
count,3.243449e+07,3.243449e+07,3.243449e+07,3.243449e+07
mean,1.710749e+06,2.557634e+04,8.351076e+00,5.896975e-01
std,9.873007e+05,1.409669e+04,7.126671e+00,4.918886e-01
min,2.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00
25%,8.559430e+05,1.353000e+04,3.000000e+00,0.000000e+00
50%,1.711048e+06,2.525600e+04,6.000000e+00,1.000000e+00
75%,2.565514e+06,3.793500e+04,1.100000e+01,1.000000e+00
max,3.421083e+06,4.968800e+04,1.450000e+02,1.000000e+00


## Merge de Dados

In [11]:
df_merged = df_order_items.merge(
    df_products[["product_id", "product_name"]],
    on="product_id",
    how="inner",
)

# Downcast: domínio cabe em tipos menores → menos RAM no driver e nos partitions.
df_merged["order_id"] = df_merged["order_id"].astype("int32")
df_merged["product_id"] = df_merged["product_id"].astype("int32")
df_merged["add_to_cart_order"] = df_merged["add_to_cart_order"].astype("int16")
df_merged["reordered"] = df_merged["reordered"].astype("int8")

print(f"Dataset final: {df_merged.shape[0]} linhas carregadas.")


Dataset final: 32434489 linhas carregadas.


In [12]:
df_merged.head()

,order_id,product_id,add_to_cart_order,reordered,product_name
0,2,33120,1,1,Organic Egg Whites
1,2,28985,2,1,Michigan Organic Kale
2,2,9327,3,0,Garlic Powder
3,2,45918,4,1,Coconut Butter
4,2,30035,5,0,Natural Sweetener


## Testes

Teste simples no select do produto

In [13]:
df_products[df_products['product_id'] == 33120]


,product_id,product_name,aisle_id,department_id
33119,33120,Organic Egg Whites,86,16


Teste simples no dataframe mergeado com select de um pedido que contem o produto anteriormente testado.

In [14]:
df_merged[df_merged['order_id'] == 2]

,order_id,product_id,add_to_cart_order,reordered,product_name
0,2,33120,1,1,Organic Egg Whites
1,2,28985,2,1,Michigan Organic Kale
2,2,9327,3,0,Garlic Powder
3,2,45918,4,1,Coconut Butter
4,2,30035,5,0,Natural Sweetener
5,2,17794,6,1,Carrots
6,2,40141,7,1,Original Unflavored Gelatine Mix
7,2,1819,8,1,All Natural No Stir Creamy Almond Butter
8,2,43668,9,0,Classic Blend Cole Slaw


In [15]:
df_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 32434489 entries, 0 to 32434488
Data columns (total 5 columns):
 #   Column             Dtype
---  ------             -----
 0   order_id           int32
 1   product_id         int32
 2   add_to_cart_order  int16
 3   reordered          int8 
 4   product_name       str  
dtypes: int16(1), int32(2), int8(1), str(1)
memory usage: 1.3 GB


## Particionando Dataset

In [16]:
# Evita string[pyarrow] no meta sem pyarrow>=13 instalado.
dask.config.set({"dataframe.convert-string": False})

# O plugin de shuffle do Distributed loga ciclo de vida em WARNING (ruído em operação normal).
logging.getLogger("distributed.shuffle").setLevel(logging.ERROR)

ddf = dd.from_pandas(df_merged, npartitions=30)
ddf = ddf.persist()
wait(ddf)

# del df_merged, df_products, df_order_items
gc.collect()

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 1.33 GiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


777

# Exercícios

## Top 10 produtos mais pedidos?

In [17]:
top_products = (
    ddf.groupby('product_id')
       .size(split_out=8)
       .nlargest(10)
)
top = top_products.compute()
products = ddf[['product_id', 'product_name']].drop_duplicates().compute()
result = top.reset_index().merge(products, on='product_id')

del top_products, top, products
gc.collect()

result

,product_id,0,product_name
0,24852,472565,Banana
1,13176,379450,Bag of Organic Bananas
2,21137,264683,Organic Strawberries
3,21903,241921,Organic Baby Spinach
4,47209,213584,Organic Hass Avocado
5,47766,176815,Organic Avocado
6,47626,152657,Large Lemon
7,16797,142951,Strawberries
8,26209,140627,Limes
9,27845,137905,Organic Whole Milk


## Quais formam o coração do e-commerce instacart?

No contexto de um dataset como o do Instacart, o Pareto de Produtos serve para identificar quais itens são o "coração" do negócio.

In [18]:
product_counts = (
    ddf.groupby('product_name')
       .size()
       .compute()
       .sort_values(ascending=False)
)
cum_pct = (product_counts.cumsum() / product_counts.sum())
del product_counts
gc.collect()

cum_pct_formatted = (cum_pct * 100).round(2)
cum_pct_formatted.head(20)

product_name
Banana                       1.46
Bag of Organic Bananas       2.63
Organic Strawberries         3.44
Organic Baby Spinach         4.19
Organic Hass Avocado         4.85
Organic Avocado              5.39
Large Lemon                  5.86
Strawberries                 6.30
Limes                        6.74
Organic Whole Milk           7.16
Organic Raspberries          7.59
Organic Yellow Onion         7.93
Organic Garlic               8.27
Organic Zucchini             8.60
Organic Blueberries          8.91
Cucumber Kirby               9.21
Organic Fuji Apple           9.48
Organic Lemon                9.75
Apple Honeycrisp Organic    10.01
Organic Grape Tomatoes      10.27
dtype: float64

Apesar de o produto Banana apresentar o maior volume absoluto e taxa de recompra, sua participação relativa no total de pedidos permanece baixa, evidenciando uma distribuição de cauda longa, característica comum em sistemas de recomendação e comércio eletrônico

## Como é a distribuição de tamanho dos pedidos?

In [19]:
order_sizes = ddf.groupby('order_id').size()
distribution = order_sizes.describe().compute()

del order_sizes
gc.collect()
distribution

count    3.214874e+06
mean     1.008888e+01
std      7.525398e+00
min      1.000000e+00
25%      5.000000e+00
50%      8.000000e+00
75%      1.400000e+01
max      1.450000e+02
dtype: float64

Mais de 3,2 milhões de pedidos únicos foram analisados.

* mean 10.08 - Em média, um pedido tem cerca de 10 itens.
* std 7.52 - O desvio padrão.
* min	1.0 -	O menor pedido teve apenas 1 item.
* 25% 5.0 - Itens no pedido
* 50%	8.0 -	Metade dos pedidos tem até 8 itens.
* 75% 14.00 - Pedidos têm até 14 itens. Apenas 25% são maiores que isso.
* max - A máxima foi 145 itens em um único no pedido.

## Qual a probabilidade de reorder(recompra) no produto?

In [20]:
product_reorder = (
    ddf.groupby('product_name')
       .agg({
           'reordered': 'mean',
           'product_name': 'count'
       })
       .rename(columns={
           'reordered': 'reorder_rate',
           'product_name': 'total_orders'
       })
       .compute()
)
product_reorder['reorder_rate'] = (product_reorder['reorder_rate'] * 100).round(2)

Top Reorder

In [21]:
top_n = 10

top_reorder = product_reorder.nlargest(top_n, 'reorder_rate').reset_index()
top_reorder

,product_name,reorder_rate,total_orders
0,Raw Veggie Wrappers,94.12,68
1,Serenity Ultimate Extrema Overnight Pads,93.10,87
2,Orange Energy Shots,92.31,13
3,Chocolate Love Bar,92.08,101
4,Soy Powder Infant Formula,91.43,35
5,Simply Sleep Nighttime Sleep Aid,91.11,45
6,"Energy Shot, Grape Flavor",90.91,22
7,Maca Buttercups,90.00,100
8,Sparking Water,90.00,60
9,Russian River Valley Reserve Pinot Noir,90.00,30


Top Reorder com filtro de mínimo de pedidos

In [22]:
min_orders = 5000

top_reorder_filtered = (
    product_reorder[product_reorder['total_orders'] >= min_orders]
    .nlargest(top_n, 'reorder_rate')
    .reset_index()
)

top_reorder_filtered


,product_name,reorder_rate,total_orders
0,Whole Organic Omega 3 Milk,86.02,9108
1,Organic Lactose Free Whole Milk,85.90,8477
2,"Milk, Organic, Vitamin D",85.43,20198
3,Organic Reduced Fat Milk,85.07,35663
4,Goat Milk,85.00,5185
5,Banana,84.35,472565
6,Organic Whole Milk,84.12,9842
7,Organic Lowfat 1% Milk,83.99,14869
8,Organic Reduced Fat Omega-3 Milk,83.83,5159
9,"Organic Milk Reduced Fat, 2% Milkfat",83.75,12737


In [23]:
del product_reorder, top_reorder, top_reorder_filtered
gc.collect()

1269

# Desafio

O desafio é encontrar produtos que são pedidos juntos, isso requer muito poder computacional a ponto que chega estourar memória caso seja feito de forma inadequada. Pois são 32 milhões de linhas vezes N.

## Quais são top produtos pedidos juntos?

In [24]:
df = ddf.drop(columns=['add_to_cart_order', 'reordered', 'product_name'])

In [25]:
df.head()

,order_id,product_id
0,2,33120
1,2,28985
2,2,9327
3,2,45918
4,2,30035


In [26]:
def pairs_per_partition(part: pd.DataFrame) -> pd.DataFrame:
    """Pares (product_id menor, maior) dentro do mesmo pedido, por partição.

    Pré-condição: índice = ``order_id`` (ex.: ``ddf.set_index('order_id')``),
    para cada pedido ficar inteiro numa partição — senão o groupby quebra pedidos.
    """
    empty = pd.DataFrame(
        {
            "p1": pd.Series(dtype="int32"),
            "p2": pd.Series(dtype="int32"),
            "count": pd.Series(dtype="int64"),
        }
    )
    if part.empty:
        return empty

    rows = []
    for _, g in part.groupby(level=0):
        prods = sorted(set(g["product_id"].tolist()))
        if len(prods) < 2:
            continue
        rows.extend(combinations(prods, 2))

    if not rows:
        return empty

    edge = pd.DataFrame(rows, columns=["p1", "p2"], dtype="int32")
    return edge.groupby(["p1", "p2"], observed=True).size().reset_index(name="count")


In [27]:
meta = pd.DataFrame(
    {"p1": pd.Series(dtype="int32"), "p2": pd.Series(dtype="int32"), "count": pd.Series(dtype="int64")}
)

ddf_basket = df[["order_id", "product_id"]].set_index("order_id")
partials = ddf_basket.map_partitions(pairs_per_partition, meta=meta)
totals = partials.groupby(["p1", "p2"])["count"].sum(split_out=16).reset_index()
top_pairs = totals.nlargest(50, "count").compute()
top_pairs = top_pairs.rename(columns={"p1": "product_id_smaller", "p2": "product_id_larger"})


In [28]:
pm = ddf[["product_id", "product_name"]].drop_duplicates(subset=["product_id"]).compute()
top_pairs_named = top_pairs.merge(
    pm, left_on="product_id_smaller", right_on="product_id", how="inner"
).drop(columns=["product_id"]).rename(columns={"product_name": "product_name_smaller"})
top_pairs_named = top_pairs_named.merge(
    pm, left_on="product_id_larger", right_on="product_id", how="inner"
).drop(columns=["product_id"]).rename(columns={"product_name": "product_name_larger"})

top_pairs_named

,product_id_smaller,product_id_larger,count,product_name_smaller,product_name_larger
0,13176,47209,62341,Bag of Organic Bananas,Organic Hass Avocado
1,13176,21137,61628,Bag of Organic Bananas,Organic Strawberries
2,21137,24852,56156,Organic Strawberries,Banana
3,24852,47766,53395,Banana,Organic Avocado
4,21903,24852,51395,Organic Baby Spinach,Banana
5,13176,21903,50372,Bag of Organic Bananas,Organic Baby Spinach
6,16797,24852,41232,Strawberries,Banana
7,24852,47626,40880,Banana,Large Lemon
8,21137,47209,40794,Organic Strawberries,Organic Hass Avocado
9,13176,27966,40503,Bag of Organic Bananas,Organic Raspberries


In [29]:
df_merged[df_merged['product_name'].str.contains('Cucumber Kirby', case=False, na=False)]

,order_id,product_id,add_to_cart_order,reordered,product_name
308,32,49683,7,1,Cucumber Kirby
416,52,49683,4,1,Cucumber Kirby
772,88,49683,20,0,Cucumber Kirby
846,95,49683,12,1,Cucumber Kirby
1012,111,49683,5,1,Cucumber Kirby
...,...,...,...,...,...
32432913,3420917,49683,21,1,Cucumber Kirby
32433136,3420935,49683,5,1,Cucumber Kirby
32433469,3420971,49683,2,1,Cucumber Kirby
32433653,3420987,49683,3,1,Cucumber Kirby


In [30]:
df_merged[df_merged['order_id'] == 32]

,order_id,product_id,add_to_cart_order,reordered,product_name
302,32,12384,1,1,Organic Lactose Free 1% Lowfat Milk
303,32,15991,2,0,Lactose Free Grade A Low Fat Vanilla Yogurt
304,32,13176,3,1,Bag of Organic Bananas
305,32,20995,4,1,Organic Broccoli Florets
306,32,18362,5,0,Organic Bread with 21 Whole Grains
307,32,35887,6,0,Organic Mixed Vegetables
308,32,49683,7,1,Cucumber Kirby
309,32,4920,8,0,Seedless Red Grapes
310,32,28199,9,1,"Clementines, Bag"
